# Projet Kayak — Notebook pur

Pipeline complet : **Nominatim → OpenWeather → Top 5 → Booking → Merge → S3 → Neon → Cartes**

> Ouvrir ce notebook depuis la racine du projet. Le fichier `.env` doit être présent localement.

## 0. Imports & configuration

In [1]:
import os
import math
import time
from pathlib import Path
from urllib.parse import quote_plus, urljoin

import pandas as pd
import requests
import folium
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import IFrame, display

ROOT = Path(".")
DATA_DIR = ROOT / "data"
RAW_HTML_DIR = DATA_DIR / "raw_booking_html"
MAPS_DIR = ROOT / "maps"

DATA_DIR.mkdir(exist_ok=True)
RAW_HTML_DIR.mkdir(exist_ok=True)
MAPS_DIR.mkdir(exist_ok=True)

load_dotenv(ROOT / ".env", override=True)  # override=True force le rechargement à chaque exécution
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
SCRAPINGBEE_API_KEY = os.getenv("SCRAPINGBEE_API_KEY")
NEON_DATABASE_URL = os.getenv("NEON_DATABASE_URL")

print("OPENWEATHER_API_KEY :", "OK" if OPENWEATHER_API_KEY else "MANQUANTE")
print("SCRAPINGBEE_API_KEY :", "OK" if SCRAPINGBEE_API_KEY else "MANQUANTE")
print("NEON_DATABASE_URL   :", "OK" if NEON_DATABASE_URL else "MANQUANTE")

OPENWEATHER_API_KEY : OK
SCRAPINGBEE_API_KEY : OK
NEON_DATABASE_URL   : OK


## 1. Géocodage Nominatim

In [2]:
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]

NOMINATIM_HEADERS = {"User-Agent": "KayakProjectBirane/1.0 (student project geocoding)"}


def geocode_city(city: str) -> tuple:
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": f"{city}, France", "format": "jsonv2", "limit": 1}
    try:
        response = requests.get(url, params=params, headers=NOMINATIM_HEADERS, timeout=30)
        http_code = response.status_code
        if response.status_code != 200:
            return {"city": city, "lat": None, "lon": None}, "KO", http_code, f"HTTP {http_code}"
        results = response.json()
        if not results:
            return {"city": city, "lat": None, "lon": None}, "KO", http_code, "Aucun résultat retourné"
        first = results[0]
        return {"city": city, "lat": float(first["lat"]), "lon": float(first["lon"])}, "OK", http_code, None
    except requests.exceptions.Timeout:
        return {"city": city, "lat": None, "lon": None}, "KO", None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return {"city": city, "lat": None, "lon": None}, "KO", None, f"ConnectionError: {e}"
    except Exception as e:
        return {"city": city, "lat": None, "lon": None}, "KO", None, str(e)


rows = []
errors = []

for idx, city in enumerate(CITIES, start=1):
    row, status, http_code, error_msg = geocode_city(city)
    rows.append(row)

    code_str = f"HTTP {http_code}" if http_code else "N/A"
    if status == "OK":
        print(f"[{idx:02d}/{len(CITIES)}] {status} ({code_str}) — {city} -> lat={row['lat']:.4f}, lon={row['lon']:.4f}")
    else:
        print(f"[{idx:02d}/{len(CITIES)}] {status} ({code_str}) — {city} -> ERREUR : {error_msg}")
        errors.append({"city": city, "http_code": http_code, "error": error_msg})

    time.sleep(1.1)

df_geo = pd.DataFrame(rows)
df_geo.insert(0, "id", range(1, len(df_geo) + 1))
df_geo.to_csv(DATA_DIR / "cities_geocoded.csv", index=False, encoding="utf-8")

print(f"\n{'='*55}")
ok_count = sum(1 for r in rows if r["lat"] is not None)
print(f"Résultat : {ok_count}/{len(CITIES)} villes géocodées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*55}")
print("Saved -> data/cities_geocoded.csv")
df_geo.head()

[01/35] OK (HTTP 200) — Mont Saint Michel -> lat=48.6360, lon=-1.5115
[02/35] OK (HTTP 200) — St Malo -> lat=48.6495, lon=-2.0260
[03/35] OK (HTTP 200) — Bayeux -> lat=49.2765, lon=-0.7025
[04/35] OK (HTTP 200) — Le Havre -> lat=49.4939, lon=0.1080
[05/35] OK (HTTP 200) — Rouen -> lat=49.4405, lon=1.0940
[06/35] OK (HTTP 200) — Paris -> lat=48.8535, lon=2.3484
[07/35] OK (HTTP 200) — Amiens -> lat=49.8942, lon=2.2957
[08/35] OK (HTTP 200) — Lille -> lat=50.6366, lon=3.0635
[09/35] OK (HTTP 200) — Strasbourg -> lat=48.5846, lon=7.7507
[10/35] OK (HTTP 200) — Chateau du Haut Koenigsbourg -> lat=48.2494, lon=7.3439
[11/35] OK (HTTP 200) — Colmar -> lat=48.0778, lon=7.3580
[12/35] OK (HTTP 200) — Eguisheim -> lat=48.0448, lon=7.3080
[13/35] OK (HTTP 200) — Besancon -> lat=47.2380, lon=6.0244
[14/35] OK (HTTP 200) — Dijon -> lat=47.3216, lon=5.0415
[15/35] OK (HTTP 200) — Annecy -> lat=45.8992, lon=6.1289
[16/35] OK (HTTP 200) — Grenoble -> lat=45.1876, lon=5.7358
[17/35] OK (HTTP 200) — Ly

,id,city,lat,lon
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


## 2. Météo OpenWeather + calcul du score

In [3]:
from collections import defaultdict

def compute_weather_score(avg_temp_7d, avg_pop_7d, total_rain_7d, avg_wind_7d) -> float:
    score = 100.0
    score -= abs(avg_temp_7d - 24) * 2.0
    score -= avg_pop_7d * 30.0
    score -= total_rain_7d * 1.5
    score -= avg_wind_7d * 0.8
    return round(score, 2)


def fetch_forecast(lat: float, lon: float) -> tuple:
    """
    Endpoint gratuit /forecast — prévisions toutes les 3h sur 5 jours (40 entrées).
    Retourne (data, http_code, error_msg).
    """
    if not OPENWEATHER_API_KEY:
        return None, None, "OPENWEATHER_API_KEY manquante dans .env"
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat": lat, "lon": lon,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
        "cnt": 40,  # 5 jours x 8 créneaux de 3h
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        http_code = response.status_code
        if response.status_code != 200:
            return None, http_code, f"HTTP {http_code} — {response.json().get('message', '')}"
        return response.json(), http_code, None
    except requests.exceptions.Timeout:
        return None, None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return None, None, f"ConnectionError: {e}"
    except Exception as e:
        return None, None, str(e)


def summarize_city_weather(city: str, lat: float, lon: float) -> tuple:
    """
    Agrège les créneaux 3h en indicateurs journaliers puis calcule le score.
    Retourne (dict_résultat, status, http_code, error_msg).
    """
    data, http_code, error_msg = fetch_forecast(lat, lon)
    if data is None:
        return None, "KO", http_code, error_msg

    # Agrégation par jour (clé = date YYYY-MM-DD)
    daily = defaultdict(lambda: {"temps": [], "pops": [], "rains": [], "winds": []})
    for slot in data.get("list", []):
        day_key = slot["dt_txt"][:10]
        daily[day_key]["temps"].append(slot["main"]["temp"])
        daily[day_key]["pops"].append(slot.get("pop", 0))
        daily[day_key]["rains"].append(slot.get("rain", {}).get("3h", 0))
        daily[day_key]["winds"].append(slot["wind"]["speed"])

    days = sorted(daily.keys())[:5]  # 5 premiers jours complets

    if len(days) < 3:
        return None, "KO", http_code, f"Pas assez de jours de prévision ({len(days)})"

    avg_temp  = round(sum(sum(daily[d]["temps"]) / len(daily[d]["temps"]) for d in days) / len(days), 2)
    avg_pop   = round(sum(sum(daily[d]["pops"])  / len(daily[d]["pops"])  for d in days) / len(days), 4)
    total_rain = round(sum(sum(daily[d]["rains"]) for d in days), 2)
    avg_wind  = round(sum(sum(daily[d]["winds"]) / len(daily[d]["winds"]) for d in days) / len(days), 2)

    result = {
        "city": city, "lat": lat, "lon": lon,
        "avg_temp_7d": avg_temp,
        "avg_pop_7d": avg_pop,
        "total_rain_7d": total_rain,
        "avg_wind_7d": avg_wind,
        "weather_score": compute_weather_score(avg_temp, avg_pop, total_rain, avg_wind),
    }
    return result, "OK", http_code, None


# --- Boucle principale ---
df_geo = pd.read_csv(DATA_DIR / "cities_geocoded.csv")
weather_rows = []
errors = []

for idx, row in df_geo.iterrows():
    city = row["city"]
    # Villes sans coordonnées (échec Nominatim)
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        print(f"[{idx+1:02d}/{len(df_geo)}] SKIP — {city} (pas de coordonnées GPS)")
        continue

    result, status, http_code, error_msg = summarize_city_weather(city, float(row["lat"]), float(row["lon"]))
    code_str = f"HTTP {http_code}" if http_code else "N/A"

    if status == "OK":
        weather_rows.append(result)
        print(f"[{idx+1:02d}/{len(df_geo)}] OK  ({code_str}) — {city} | score={result['weather_score']} | temp={result['avg_temp_7d']}°C")
    else:
        print(f"[{idx+1:02d}/{len(df_geo)}] KO  ({code_str}) — {city} -> ERREUR : {error_msg}")
        errors.append({"city": city, "http_code": http_code, "error": error_msg})

df_weather = pd.DataFrame(weather_rows)
df_weather.to_csv(DATA_DIR / "weather.csv", index=False, encoding="utf-8")

print(f"\n{'='*60}")
print(f"Résultat : {len(weather_rows)}/{len(df_geo)} villes météo récupérées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*60}")
print("Saved -> data/weather.csv")
df_weather[["city", "avg_temp_7d", "avg_pop_7d", "total_rain_7d", "avg_wind_7d", "weather_score"]].head(10)

[01/35] OK  (HTTP 200) — Mont Saint Michel | score=62.82 | temp=15.96°C
[02/35] OK  (HTTP 200) — St Malo | score=66.06 | temp=15.65°C
[03/35] OK  (HTTP 200) — Bayeux | score=66.54 | temp=15.72°C
[04/35] OK  (HTTP 200) — Le Havre | score=63.78 | temp=15.1°C
[05/35] OK  (HTTP 200) — Rouen | score=58.51 | temp=15.75°C
[06/35] OK  (HTTP 200) — Paris | score=72.43 | temp=16.4°C
[07/35] OK  (HTTP 200) — Amiens | score=55.46 | temp=15.1°C
[08/35] OK  (HTTP 200) — Lille | score=66.46 | temp=15.38°C
[09/35] OK  (HTTP 200) — Strasbourg | score=68.23 | temp=15.0°C
[10/35] OK  (HTTP 200) — Chateau du Haut Koenigsbourg | score=65.96 | temp=12.91°C
[11/35] OK  (HTTP 200) — Colmar | score=69.06 | temp=15.68°C
[12/35] OK  (HTTP 200) — Eguisheim | score=69.59 | temp=15.46°C
[13/35] OK  (HTTP 200) — Besancon | score=65.82 | temp=14.53°C
[14/35] OK  (HTTP 200) — Dijon | score=68.07 | temp=14.71°C
[15/35] OK  (HTTP 200) — Annecy | score=72.77 | temp=15.78°C
[16/35] OK  (HTTP 200) — Grenoble | score=77.52 

,city,avg_temp_7d,avg_pop_7d,total_rain_7d,avg_wind_7d,weather_score
0,Mont Saint Michel,15.96,0.2704,6.15,4.70,62.82
1,St Malo,15.65,0.2302,4.19,5.06,66.06
2,Bayeux,15.72,0.1951,5.08,4.28,66.54
3,Le Havre,15.10,0.1671,6.64,4.31,63.78
4,Rouen,15.75,0.3404,8.17,3.16,58.51
5,Paris,16.40,0.1858,3.10,2.68,72.43
6,Amiens,15.10,0.3423,9.23,3.28,55.46
7,Lille,15.38,0.2301,4.42,3.46,66.46
8,Strasbourg,15.00,0.2042,4.06,1.94,68.23
9,Chateau du Haut Koenigsbourg,12.91,0.1692,3.64,1.65,65.96


## 3. Sélection du Top 5

In [4]:
df_weather = pd.read_csv(DATA_DIR / "weather.csv")

df_top = (
    df_weather
    .sort_values(by="weather_score", ascending=False)
    .head(5)
    .copy()
)

df_top.to_csv(DATA_DIR / "top_cities.csv", index=False, encoding="utf-8")

print("Saved -> data/top_cities.csv")
df_top[["city", "weather_score"]]

Saved -> data/top_cities.csv


,city,weather_score
24,Nimes,89.86
28,Carcassonne,89.66
33,Bayonne,89.56
21,Aix en Provence,89.17
22,Avignon,89.12


## 4. Scraping Booking (ScrapingBee)

In [5]:
# Test rapide de la clé ScrapingBee avant de lancer le scraping
load_dotenv(ROOT / ".env", override=True)
SCRAPINGBEE_API_KEY = os.getenv("SCRAPINGBEE_API_KEY")
print(f"Clé chargée : {SCRAPINGBEE_API_KEY[:10]}...{SCRAPINGBEE_API_KEY[-6:]}")

test_url = "https://www.booking.com/searchresults.fr.html?ss=Paris"
params_test = {
    "api_key": SCRAPINGBEE_API_KEY,
    "url": test_url,
    "render_js": "true",
    "stealth_proxy": "true",
    "country_code": "fr",
    "block_resources": "false",
}

try:
    r = requests.get("https://app.scrapingbee.com/api/v1/", params=params_test, timeout=90)
    http_code = r.status_code
    if http_code == 200:
        credits_used = r.headers.get("Spb-cost", "?")
        credits_left = r.headers.get("Spb-remaining-api-credits", "?")
        cards = r.text.count('data-testid="property-card"')
        print(f"[SCRAPINGBEE] OK (HTTP 200)")
        print(f"  Crédits utilisés : {credits_used} | Crédits restants : {credits_left}")
        print(f"  property-cards détectées : {cards}")
    else:
        try:
            detail = r.json()
        except Exception:
            detail = r.text[:300]
        print(f"[SCRAPINGBEE] KO (HTTP {http_code}) — {detail}")
except requests.exceptions.Timeout:
    print("[SCRAPINGBEE] KO — Timeout")
except Exception as e:
    print(f"[SCRAPINGBEE] KO — {e}")

Clé chargée : 63Y84FOSD9...BPXSZE
[SCRAPINGBEE] OK (HTTP 200)
  Crédits utilisés : 75 | Crédits restants : ?
  property-cards détectées : 25


In [6]:
def build_booking_url_variants(city: str) -> list:
    city_simple = quote_plus(city)
    city_france = quote_plus(f"{city}, France")
    return [
        ("city_france",      f"https://www.booking.com/searchresults.fr.html?ss={city_france}"),
        ("city_full",        f"https://www.booking.com/searchresults.fr.html?ss={city_simple}&lang=fr&group_adults=2&no_rooms=1&group_children=0"),
        ("city_simple",      f"https://www.booking.com/searchresults.fr.html?ss={city_simple}"),
        ("city_france_full", f"https://www.booking.com/searchresults.fr.html?ss={city_france}&lang=fr&group_adults=2&no_rooms=1&group_children=0"),
    ]


def fetch_html_scrapingbee(target_url: str) -> tuple:
    """
    Retourne (html, http_code, error_msg).
    Ne lève jamais d'exception.
    """
    if not SCRAPINGBEE_API_KEY:
        return None, None, "SCRAPINGBEE_API_KEY manquante dans .env"
    params = {
        "api_key": SCRAPINGBEE_API_KEY,
        "url": target_url,
        "stealth_proxy": "true",
        "country_code": "fr",
        "render_js": "true",
        "block_resources": "false",
    }
    try:
        response = requests.get("https://app.scrapingbee.com/api/v1/", params=params, timeout=90)
        http_code = response.status_code
        if response.status_code != 200:
            # ScrapingBee renvoie un message JSON ou texte en cas d'erreur
            try:
                detail = response.json().get("message", response.text[:200])
            except Exception:
                detail = response.text[:200]
            return None, http_code, f"HTTP {http_code} — {detail}"
        return response.text, http_code, None
    except requests.exceptions.Timeout:
        return None, None, "Timeout"
    except requests.exceptions.ConnectionError as e:
        return None, None, f"ConnectionError: {e}"
    except Exception as e:
        return None, None, str(e)


def html_has_property_cards(html: str):
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.get_text(" ", strip=True) if soup.title else "NO TITLE"
    count = html.count('data-testid="property-card"')
    return count > 0, count, title


def save_city_html(city: str, html: str) -> Path:
    safe = (
        city.lower()
        .replace(" ", "_").replace("é", "e").replace("è", "e").replace("ê", "e")
        .replace("à", "a").replace("ù", "u").replace("î", "i").replace("ï", "i")
        .replace("ô", "o").replace("ç", "c").replace("'", "").replace("-", "_")
    )
    path = RAW_HTML_DIR / f"booking_{safe}.html"
    path.write_text(html, encoding="utf-8")
    return path


df_top = pd.read_csv(DATA_DIR / "top_cities.csv")
errors = []

for idx, row in df_top.iterrows():
    city = row["city"]
    print(f"\n[BOOKING] {idx + 1}/{len(df_top)} -> {city}")
    variants = build_booking_url_variants(city)
    html_saved, used_variant = None, None

    for label, url in variants:
        print(f"  [TRY] variant={label}")
        html, http_code, error_msg = fetch_html_scrapingbee(url)
        code_str = f"HTTP {http_code}" if http_code else "N/A"

        if html is None:
            print(f"  [KO]  ({code_str}) variant={label} -> {error_msg}")
            # Erreur bloquante (401 clé invalide, timeout...) : inutile d'essayer les autres variantes
            if http_code in (401, 403) or http_code is None:
                errors.append({"city": city, "variant": label, "http_code": http_code, "error": error_msg})
                break
            continue

        ok, count, title = html_has_property_cards(html)
        print(f"  [OK]  ({code_str}) variant={label} | property_cards={count} | title={title}")

        if ok:
            html_saved = html
            used_variant = label
            break
        else:
            print(f"  [SKIP] variant={label} — page sans résultats hôtels")

    if html_saved is None:
        msg = f"Aucune variante valide pour {city}"
        print(f"  [WARN] {msg}")
        errors.append({"city": city, "variant": "all", "http_code": None, "error": msg})
        continue

    saved_path = save_city_html(city, html_saved)
    print(f"  [SAVED] {saved_path} (variant={used_variant})")

print(f"\n{'='*60}")
ok_count = len(df_top) - len({e["city"] for e in errors})
print(f"Résultat : {ok_count}/{len(df_top)} villes scrapées avec succès")
if errors:
    print(f"Erreurs ({len(errors)}) :")
    for e in errors:
        print(f"  - {e['city']} | variant={e['variant']} | code={e['http_code']} | {e['error']}")
else:
    print("Aucune erreur.")
print(f"{'='*60}")


[BOOKING] 1/5 -> Nimes
  [TRY] variant=city_france
  [OK]  (HTTP 200) variant=city_france | property_cards=25 | title=Booking.com:
Hôtels : Nîmes.
Réservez votre hôtel dès maintenant !
  [SAVED] data\raw_booking_html\booking_nimes.html (variant=city_france)

[BOOKING] 2/5 -> Carcassonne
  [TRY] variant=city_france
  [OK]  (HTTP 200) variant=city_france | property_cards=25 | title=Booking.com:
Hôtels : Carcassonne.
Réservez votre hôtel dès maintenant !
  [SAVED] data\raw_booking_html\booking_carcassonne.html (variant=city_france)

[BOOKING] 3/5 -> Bayonne
  [TRY] variant=city_france
  [OK]  (HTTP 200) variant=city_france | property_cards=0 | title=Booking.com | Site officiel | Hôtels, vols, voitures de location et hébergements
  [SKIP] variant=city_france — page sans résultats hôtels
  [TRY] variant=city_full
  [OK]  (HTTP 200) variant=city_full | property_cards=25 | title=Booking.com:
Hôtels : Bayonne.
Réservez votre hôtel dès maintenant !
  [SAVED] data\raw_booking_html\booking_bayon

## 5. Parsing Booking → CSV hôtels

In [7]:
FILENAME_TO_CITY = {
    "booking_avignon.html":          "Avignon",
    "booking_aix_en_provence.html":  "Aix en Provence",
    "booking_nimes.html":            "Nimes",
    "booking_bormes_les_mimosas.html": "Bormes les Mimosas",
    "booking_strasbourg.html":       "Strasbourg",
}


def extract_score_from_card(card):
    score_tag = card.select_one('[data-testid="review-score"]')
    if score_tag:
        text = score_tag.get_text(" ", strip=True).replace(",", ".")
        for token in text.split():
            try:
                v = float(token)
                if 0 <= v <= 10:
                    return v
            except Exception:
                pass
    for tag in card.select('[data-testid="review-score"] div, [aria-label*="Note"], [aria-label*="Scored"]'):
        text = tag.get_text(" ", strip=True).replace(",", ".")
        for token in text.split():
            try:
                v = float(token)
                if 0 <= v <= 10:
                    return v
            except Exception:
                pass
    return None


def extract_name_from_card(card):
    for sel in ('[data-testid="title"]', "div.f6431b446c", "div[data-testid='title'] div"):
        tag = card.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt and len(txt) > 2:
                return txt
    link = card.select_one('a[href*="/hotel/"]')
    if link:
        txt = link.get_text(" ", strip=True)
        if txt and len(txt) > 2:
            return txt
    return None


def extract_link_from_card(card):
    link = card.select_one('a[href*="/hotel/"]')
    if not link:
        return None
    href = link.get("href")
    return urljoin("https://www.booking.com", href) if href else None


def extract_description_from_card(card):
    for sel in (
        '[data-testid="property-card-unit-configuration"]',
        '[data-testid="property-card-unit-configuration-group"]',
        'div.abf093bdfe', 'div.c624d7469d',
    ):
        tag = card.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt and len(txt) > 10:
                return txt
    return None


def parse_booking_hotels(html: str, city: str, max_hotels: int = 10) -> list:
    soup = BeautifulSoup(html, "lxml")
    cards = soup.select('div[data-testid="property-card"]')
    print(f"  [PARSE] {city} — {len(cards)} property-cards trouvées")
    rows, seen = [], set()
    for card in cards:
        name = extract_name_from_card(card)
        url  = extract_link_from_card(card)
        if not name or not url or url in seen:
            continue
        seen.add(url)
        rows.append({
            "city": city, "hotel_name": name,
            "hotel_score": extract_score_from_card(card),
            "url": url,
            "description": extract_description_from_card(card),
        })
        if len(rows) >= max_hotels:
            break
    return rows


all_rows = []
for path in sorted(RAW_HTML_DIR.glob("booking_*.html")):
    city = FILENAME_TO_CITY.get(path.name, path.stem.replace("booking_", "").replace("_", " ").title())
    print(f"\n[FILE] {path.name} -> {city}")
    html = path.read_text(encoding="utf-8", errors="ignore")
    rows = parse_booking_hotels(html, city=city)
    print(f"  -> {len(rows)} hôtels extraits")
    all_rows.extend(rows)

df_hotels = pd.DataFrame(all_rows)
df_hotels.to_csv(DATA_DIR / "hotels_multi_cities.csv", index=False, encoding="utf-8")

print("\nSaved -> data/hotels_multi_cities.csv")
print(df_hotels["city"].value_counts())
df_hotels.head()


[FILE] booking_aix_en_provence.html -> Aix en Provence
  [PARSE] Aix en Provence — 25 property-cards trouvées
  -> 10 hôtels extraits

[FILE] booking_avignon.html -> Avignon
  [PARSE] Avignon — 25 property-cards trouvées
  -> 10 hôtels extraits

[FILE] booking_bayonne.html -> Bayonne
  [PARSE] Bayonne — 25 property-cards trouvées
  -> 10 hôtels extraits

[FILE] booking_carcassonne.html -> Carcassonne
  [PARSE] Carcassonne — 25 property-cards trouvées
  -> 10 hôtels extraits

[FILE] booking_nimes.html -> Nimes
  [PARSE] Nimes — 25 property-cards trouvées
  -> 10 hôtels extraits

Saved -> data/hotels_multi_cities.csv
city
Aix en Provence    10
Avignon            10
Bayonne            10
Carcassonne        10
Nimes              10
Name: count, dtype: int64


,city,hotel_name,hotel_score,url,description
0,Aix en Provence,Séjours & Affaires Aix-en-Provence Mirabeau,8.1,https://www.booking.com/hotel/fr/residence-mir...,None
1,Aix en Provence,Appart'hôtel Odalys City - Aix en Provence Cen...,8.0,https://www.booking.com/hotel/fr/atrium-d-anai...,None
2,Aix en Provence,Appartement des Augustins,9.2,https://www.booking.com/hotel/fr/appart-des-au...,None
3,Aix en Provence,Hôtel Paul,8.6,https://www.booking.com/hotel/fr/paul.fr.html?...,None
4,Aix en Provence,Hôtel des Augustins,8.4,https://www.booking.com/hotel/fr/les-augustins...,None


## 6. Fusion hôtels + météo → dataset final

In [8]:
df_weather = pd.read_csv(DATA_DIR / "weather.csv")
df_hotels  = pd.read_csv(DATA_DIR / "hotels_multi_cities.csv")

cities_scraped = set(df_hotels["city"].dropna().unique())
df_weather_top = df_weather[df_weather["city"].isin(cities_scraped)].copy()

df_final = df_hotels.merge(df_weather_top, on="city", how="left")
df_final.to_csv(DATA_DIR / "final_kayak_results.csv", index=False, encoding="utf-8")

print("Saved -> data/final_kayak_results.csv")
print("Shape :", df_final.shape)
print(df_final["city"].value_counts())
df_final.head()

Saved -> data/final_kayak_results.csv
Shape : (50, 12)
city
Aix en Provence    10
Avignon            10
Bayonne            10
Carcassonne        10
Nimes              10
Name: count, dtype: int64


,city,hotel_name,hotel_score,url,description,lat,lon,avg_temp_7d,avg_pop_7d,total_rain_7d,avg_wind_7d,weather_score
0,Aix en Provence,Séjours & Affaires Aix-en-Provence Mirabeau,8.1,https://www.booking.com/hotel/fr/residence-mir...,NaN,43.529842,5.447474,19.95,0.0,0.0,3.41,89.17
1,Aix en Provence,Appart'hôtel Odalys City - Aix en Provence Cen...,8.0,https://www.booking.com/hotel/fr/atrium-d-anai...,NaN,43.529842,5.447474,19.95,0.0,0.0,3.41,89.17
2,Aix en Provence,Appartement des Augustins,9.2,https://www.booking.com/hotel/fr/appart-des-au...,NaN,43.529842,5.447474,19.95,0.0,0.0,3.41,89.17
3,Aix en Provence,Hôtel Paul,8.6,https://www.booking.com/hotel/fr/paul.fr.html?...,NaN,43.529842,5.447474,19.95,0.0,0.0,3.41,89.17
4,Aix en Provence,Hôtel des Augustins,8.4,https://www.booking.com/hotel/fr/les-augustins...,NaN,43.529842,5.447474,19.95,0.0,0.0,3.41,89.17


## 7. Chargement dans Neon / PostgreSQL

In [9]:
import psycopg2
from psycopg2.extras import execute_values

EXPECTED_COLUMNS = [
    "city", "hotel_name", "hotel_score", "url", "description",
    "lat", "lon", "avg_temp_7d", "avg_pop_7d", "total_rain_7d",
    "avg_wind_7d", "weather_score",
]

if not NEON_DATABASE_URL:
    print("NEON_DATABASE_URL manquante — étape ignorée")
else:
    df_load = pd.read_csv(DATA_DIR / "final_kayak_results.csv")[EXPECTED_COLUMNS].copy()

    rows = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in df_load.itertuples(index=False, name=None)
    ]

    conn = psycopg2.connect(NEON_DATABASE_URL)
    conn.autocommit = False
    try:
        with conn.cursor() as cur:
            cur.execute("""
                DROP TABLE IF EXISTS kayak_results;
                CREATE TABLE kayak_results (
                    city TEXT, hotel_name TEXT, hotel_score DOUBLE PRECISION,
                    url TEXT, description TEXT,
                    lat DOUBLE PRECISION, lon DOUBLE PRECISION,
                    avg_temp_7d DOUBLE PRECISION, avg_pop_7d DOUBLE PRECISION,
                    total_rain_7d DOUBLE PRECISION, avg_wind_7d DOUBLE PRECISION,
                    weather_score DOUBLE PRECISION
                );
            """)
            execute_values(
                cur,
                """INSERT INTO kayak_results (city, hotel_name, hotel_score, url, description,
                   lat, lon, avg_temp_7d, avg_pop_7d, total_rain_7d, avg_wind_7d, weather_score)
                   VALUES %s""",
                rows, page_size=100,
            )
            conn.commit()
        print(f"[NEON] kayak_results chargée — {len(rows)} lignes")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

[NEON] kayak_results chargée — 50 lignes


## 8. Génération des cartes Folium

In [10]:
def clean_text(x):
    return "" if pd.isna(x) else str(x).strip()

def shorten(text, max_len=42):
    text = clean_text(text)
    return text if len(text) <= max_len else text[:max_len - 3] + "..."

def prepare_visual_coordinates(df):
    parts = []
    for city, group in df.groupby("city", sort=False):
        g = group.copy().reset_index(drop=True)
        base_lat, base_lon = float(g["lat"].iloc[0]), float(g["lon"].iloc[0])
        n = len(g)
        for i in range(n):
            if n == 1:
                g.loc[i, "plot_lat"] = base_lat
                g.loc[i, "plot_lon"] = base_lon
            else:
                angle = (2 * math.pi * i) / n
                radius = 0.025 + (0.004 * (i % 3))
                g.loc[i, "plot_lat"] = base_lat + radius * math.sin(angle)
                g.loc[i, "plot_lon"] = base_lon + (radius * math.cos(angle)) / max(math.cos(math.radians(base_lat)), 0.35)
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


# --- Carte Top 5 destinations ---
df_top5 = pd.read_csv(DATA_DIR / "top_cities.csv")
for col in ["lat", "lon", "weather_score", "avg_temp_7d", "avg_pop_7d", "total_rain_7d", "avg_wind_7d"]:
    df_top5[col] = pd.to_numeric(df_top5[col], errors="coerce")
df_top5 = df_top5.sort_values("weather_score", ascending=False).head(5)

m1 = folium.Map(location=[df_top5["lat"].mean(), df_top5["lon"].mean()], zoom_start=6)
m1.get_root().html.add_child(folium.Element('<h3 align="center" style="font-size:20px;"><b>Top 5 destinations</b></h3>'))

for _, row in df_top5.iterrows():
    popup_html = (
        f"<b>{row['city']}</b><br>"
        f"Score : {row['weather_score']:.2f}<br>"
        f"Temp. : {row['avg_temp_7d']:.1f} °C<br>"
        f"Pluie : {row['total_rain_7d']:.1f} mm<br>"
        f"Vent : {row['avg_wind_7d']:.1f} m/s"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=10,
        color="blue", fill=True, fill_color="blue", fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row["city"]
    ).add_to(m1)

m1.fit_bounds([[r["lat"], r["lon"]] for _, r in df_top5.iterrows()], padding=(40, 40))
m1.save(str(MAPS_DIR / "map_top_5_destinations.html"))
print("Saved -> maps/map_top_5_destinations.html")


# --- Carte Top 20 hôtels ---
COLOR_MAP = {
    "Aix en Provence": "blue",
    "Avignon": "green",
    "Bormes les Mimosas": "red",
    "Nimes": "purple",
    "Strasbourg": "orange",
}

df_final = pd.read_csv(DATA_DIR / "final_kayak_results.csv")
df_final["lat"] = pd.to_numeric(df_final["lat"], errors="coerce")
df_final["lon"] = pd.to_numeric(df_final["lon"], errors="coerce")
df_final["hotel_score"] = pd.to_numeric(df_final["hotel_score"], errors="coerce").fillna(0)

df_top20 = df_final.sort_values("hotel_score", ascending=False).head(20).reset_index(drop=True)
df_top20["rank"] = range(1, len(df_top20) + 1)
df_top20 = prepare_visual_coordinates(df_top20)

m2 = folium.Map(location=[df_top20["plot_lat"].mean(), df_top20["plot_lon"].mean()], zoom_start=6)
m2.get_root().html.add_child(folium.Element('<h3 align="center" style="font-size:20px;"><b>Top 20 hôtels</b></h3>'))

for _, row in df_top20.iterrows():
    city, hotel = clean_text(row["city"]), clean_text(row["hotel_name"])
    color = COLOR_MAP.get(city, "cadetblue")
    rank = int(row["rank"])
    popup_html = (
        f"<b>#{rank} {hotel}</b><br>"
        f"Ville : {city}<br>"
        f"Score : {row['hotel_score']}<br>"
        f"<a href='{clean_text(row['url'])}' target='_blank'>Booking</a>"
    )
    folium.CircleMarker(
        location=[row["plot_lat"], row["plot_lon"]], radius=10,
        color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{rank}. {hotel}"
    ).add_to(m2)
    folium.Marker(
        location=[row["plot_lat"], row["plot_lon"]],
        icon=folium.DivIcon(html=f'<div style="font-size:11px;font-weight:bold;text-align:center;width:18px;height:18px;line-height:18px;background:white;border:1px solid black;border-radius:50%;">{rank}</div>')
    ).add_to(m2)

legend_html = """
<div style="position:fixed;bottom:35px;left:35px;width:220px;background:white;
    border:2px solid grey;z-index:9999;font-size:14px;padding:10px;">
    <b>Légende</b><br>
    <span style="color:blue;">●</span> Aix en Provence<br>
    <span style="color:green;">●</span> Avignon<br>
    <span style="color:red;">●</span> Bormes les Mimosas<br>
    <span style="color:purple;">●</span> Nimes<br>
    <span style="color:orange;">●</span> Strasbourg
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html))

list_items = "".join(
    f'<li style="margin-bottom:8px;"><b>{shorten(r["hotel_name"])}</b><br>'
    f'<span style="color:#555;">{r["city"]} – {r["hotel_score"]}</span></li>'
    for _, r in df_top20.sort_values("rank").iterrows()
)
panel_html = f"""
<div style="position:fixed;top:90px;right:18px;width:320px;max-height:72vh;overflow-y:auto;
    background:rgba(255,255,255,0.96);border:1px solid #999;z-index:9999;
    padding:12px;font-size:13px;border-radius:6px;">
    <h4 style="margin:0 0 10px 0;">Top 20 hôtels</h4>
    <ol style="padding-left:22px;margin:0;">{list_items}</ol>
</div>"""
m2.get_root().html.add_child(folium.Element(panel_html))
m2.fit_bounds([[r["plot_lat"], r["plot_lon"]] for _, r in df_top20.iterrows()], padding=(40, 40))
m2.save(str(MAPS_DIR / "map_top_20_hotels.html"))
print("Saved -> maps/map_top_20_hotels.html")

Saved -> maps/map_top_5_destinations.html
Saved -> maps/map_top_20_hotels.html


## 9. Affichage des cartes

In [11]:
map1 = MAPS_DIR / "map_top_5_destinations.html"
if map1.exists():
    display(IFrame(src=str(map1), width=1200, height=500))
else:
    print("Carte Top 5 introuvable :", map1)

In [12]:
map2 = MAPS_DIR / "map_top_20_hotels.html"
if map2.exists():
    display(IFrame(src=str(map2), width=1200, height=750))
else:
    print("Carte Top 20 hôtels introuvable :", map2)